# KPI Dependency Hierarchy
This notebook displays the full dependency hierarchy for all metrics based on the extracted `depends_on` relationships.

## 1. Load Data

In [ ]:
import json
from pathlib import Path

file_path = Path('data/raw/extracted_kpis.json')
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

kpis = data.get('kpis', []) if isinstance(data, dict) else data
kpi_dict = {k['kpi_id']: k for k in kpis}
print(f'Loaded {len(kpis)} KPIs.')

## 2. KPI Dependency Hierarchy

In [ ]:
def print_hierarchy(kpi_id, level=0, visited=None):
    if visited is None:
        visited = set()
    
    kpi = kpi_dict.get(kpi_id)
    if not kpi:
        return

    name = kpi.get('name', kpi_id)
    depends_on = kpi.get('depends_on', [])
    
    # Indent based on depth (2 spaces per level)
    indent = "  " * level
    branch = "└─ " if level > 0 else ""
    
    # Print Name (ID)
    print(f"{indent}{branch}{name} ({kpi_id})")

    # Stop recursion if circular
    if kpi_id in visited:
        return
    visited.add(kpi_id)

    for dep_id in depends_on:
        # Maintain standard hierarchy indentation
        print_hierarchy(dep_id, level + 1, visited.copy())

# Calculate root nodes: any KPI that is NOT a dependency of another KPI
all_dependencies = {dep for k in kpis for dep in k.get('depends_on', [])}
roots = sorted([k for k in kpis if k['kpi_id'] not in all_dependencies], key=lambda x: x['kpi_id'])

print("KPI DEPENDENCY HIERARCHY")
print("=" * 30)

for root in roots:
    print_hierarchy(root['kpi_id'])
    if root.get('depends_on'):
        print() # Extra space after nested trees